# S6E8 Addiction Blend — LB 0.97117（解説付き写し）

- **コンペ**: [Predicting Smartphone Addiction (Playground Series S6E8)](https://www.kaggle.com/competitions/playground-series-s6e8)
- **元notebook**: [S6E8 Addiction Blend LB 0.97117](https://www.kaggle.com/code/thisray/s6e8-addiction-blend-lb-0-97117)
- **原著者**: thisray
- **スコア**: Public 0.97117（本コンペの公開最高帯）
- **作成日**: 2026-08-22

> ⚠️ これは**学習目的の解説付き写し**です。原著者のコードは変更していませんが、実行結果は含んでいません。

## 手法の概要

**6人の作者が公開した85本の予測配列 + 自作1列** を、重み付きの **rank blend（順位ブレンド）** で混ぜるだけのnotebookです。モデルの学習コードは1行もありません。にもかかわらず public LB 0.97117 と、単体モデル（0.968前後）を大きく上回ります。

このnotebookの学習価値は「高スコアを出す方法」ではなく、**大規模ブレンドの配管がどう組まれているか**、そして **なぜこのスコアが危ういのか** の2点にあります（後者は本ルーティンの 2026-08-20 分「Why Every S6E8 Notebook Above 0.97110 Overfits」と正面から対立します — 本notebookのスコア 0.97117 は、まさにその批判の射程内です）。

## 評価指標

- **タスク**: 表形式データから `addicted_label`（スマートフォン依存かどうか）を予測する**二値分類**。
- **指標**: **ROC-AUC**。「無作為に選んだ陽性サンプルのスコアが、無作為に選んだ陰性サンプルのスコアより高い確率」に等しい値です。0.5 がランダム、1.0 が完璧。
- **なぜこの指標か**: 依存傾向のようなラベルは陽性・陰性の比率が偏りがちで、かつ「何%以上を陽性と呼ぶか」という閾値は用途によって変わります。AUC は**閾値に依存せず、クラス比率の変化にも期待値として不変**なので、こうした課題に向いています。
- **この手法が指標をどう最適化しているか**: ここが本notebookの核心です。**AUC は予測値の「順位」しか見ません**。したがって
  1. 確率の絶対値を合わせる**キャリブレーションは無意味**（単調変換に対して不変なので）。
  2. モデルを混ぜるとき、**生の確率を平均すると「自信過剰なモデル」が支配的になる**。あるモデルが 0.99 を出し別のモデルが 0.6 を出したら、平均は前者に引っ張られますが、それは前者が正しいことを意味しません。
  3. 一方 **順位パーセンタイルに変換してから平均すれば、全モデルが同じスケール（0〜1の一様分布）になり、対等に混ざる**。しかも順位はAUCが読む情報そのものなので、情報の損失もない。
  
  → だから `rank_pct()` を全部にかけてから重み付き和を取る、という設計になっています。**指標の数学的性質から後処理の形が一意に決まる**、良い例です。


# S6E8 Addiction Blend — LB 0.97117

A weighted rank blend of **85 public arrays from 6 authors** plus **one column of my own**.

## Data sources

| Author | Dataset | Arrays used |
|---|---|---|
| Szymon Kłapiński | [S6E8 OOF library (47 models)](https://www.kaggle.com/datasets/szymonkapiski/s6e8-oof-library-47-models) | 52 |
| AdarshAleti | [S6E8 Adarsh OOF library](https://www.kaggle.com/datasets/adarsh1077/s6e8-adarsh-oof-library) | 20 |
| Darius Hafshar | [S6E8 OOF library - 7 models, frozen folds](https://www.kaggle.com/datasets/dariushafshar/s6e8-golem-oof-library) | 6 |
| Rayk Kretzschmar | [S6E8 factorization-machine lattice members](https://www.kaggle.com/datasets/raykkretzschmar/s6e8-fm-lattice-blend-members) | 5 |
| boltuzamaki | [S6E8 OOF Prediction Library](https://www.kaggle.com/datasets/boltuzamaki/s6e8-oof-prediction-library) | 1 |
| Naji | [Playground S6E8 - OOF & Submission](https://www.kaggle.com/datasets/najiama/predicting-smartphone-addiction-oof-submission-csv) | 1 |
| *(mine)* | [S6E8 Our Component](https://www.kaggle.com/datasets/thisray/s6e8-our-component) | 1 |

## Credits

Thank you to everyone above for sharing their work.

Factorization-machine lattice members for Playground Series S6E8 by Rayk Kretzschmar, used under Apache License 2.0.

*If this helps your ensemble, an upvote is appreciated.* 👍


## 【解説】入力データセットのパス解決と提出IDの読み込み

**What**: `root(slug)` という小さな関数で、Kaggleの入力データセットが実際にどこにマウントされたかを探します。そのうえで、コンペ本体と6つのOOFライブラリ（他の参加者が公開した予測配列の詰め合わせ）のパスを確定させ、`sample_submission.csv` から**提出すべきIDの並び順**を取得します。

**Why**:

- **パス解決を関数化する理由**: Kaggleではエディタ実行時は `/kaggle/input/<slug>`、API経由では `/kaggle/input/datasets/<user>/<slug>` にマウントされます。`glob` でパターンマッチして最初に見つかったものを返すことで、どちらでも動くようにしています。見つからなければ `FileNotFoundError` を**明示的に投げる**のも重要で、パスが無いのに空のDataFrameで先に進んでしまうと、原因不明の壊れた提出ができあがります。
- **`sub_ids` を最初に取る理由**: 85本の配列は作者ごとにファイル形式（`.npy` / `.parquet` / `.csv`）も行の並び順もバラバラです。**共通の基準となる並び順を1つ決めて、全部をそれに揃える**というのが、複数ソースを混ぜるときの鉄則。ここでは `sample_submission.csv` の順序を基準にしています。

**初心者向け補足**: **OOF (Out-Of-Fold) 予測**とは、k分割交差検証の各foldで「学習に使わなかった側」に対して行った予測のこと。訓練データ全体分の「リークしていない予測値」が手に入るので、これを新しい特徴量としてメタモデルに食わせる（＝スタッキング）ことができます。ブレンド重みの決定にも使えます。

In [ ]:
import glob

import numpy as np
import pandas as pd


def root(slug):
    """Kaggle mounts inputs at /kaggle/input/<slug> in the editor and at
    /kaggle/input/{datasets,competitions}/... for API-pushed kernels. Accept both."""
    for pattern in (f"/kaggle/input/{slug}",
                    f"/kaggle/input/datasets/*/{slug}",
                    f"/kaggle/input/competitions/{slug}"):
        hit = glob.glob(pattern)
        if hit:
            return hit[0]
    raise FileNotFoundError(slug)


COMP = root("playground-series-s6e8")
SZ   = root("s6e8-oof-library-47-models") + "/oof"
AD   = root("s6e8-adarsh-oof-library")
GO   = root("s6e8-golem-oof-library")
RK   = root("s6e8-fm-lattice-blend-members")
BO   = root("s6e8-oof-prediction-library")
NA   = root("predicting-smartphone-addiction-oof-submission-csv")
OURS = root("s6e8-our-component")

sub_ids = pd.read_csv(f"{COMP}/sample_submission.csv")["id"].to_numpy()


## 【解説】85本すべての重み — ハードコードされた `WEIGHTS` 辞書

**What**: 各予測配列に掛ける重みが辞書としてベタ書きされています。`"szymon__lat_lgbm": -1.0144...` のように、**負の重み**が多数含まれているのが目を引きます。

**Why**:

- **なぜ重みがベタ書きなのか**: この重みは別の場所（原著者のローカルまたは非公開notebook）で、OOF予測に対して**線形回帰（おそらくリッジ回帰）を当てて求めた係数**です。提出用notebookでは最適化を再実行せず、結果の数値だけを持ち込んでいます。実行時間が20秒で済む理由でもあります。
- **なぜ負の重みが出るのか**: 相関の強いモデルを大量に混ぜると、線形回帰は**多重共線性**の状態になります。ほぼ同じ情報を持つ2つの特徴量に対し、回帰は「+1.3 と -1.0」のような大きさの打ち消し合う係数を割り当てることがあり、これは実質的に**2つのモデルの差分（＝片方が他方より高く評価した部分）**を特徴量として使っていることになります。うまくいけば微細な情報を拾えますが、
- **危険な点**: 打ち消し合う大きな係数は、**OOFのノイズに過剰適合している可能性が高い**。`szymon__latr1_xgb: +1.303` と `szymon__lat_lgbm: -1.014` のペアは、OOF上ではスコアを上げても、テスト分布がわずかに違えば逆に効きます。リッジ回帰の正則化を強めれば係数は縮みますが、その分OOFスコアも下がるので、**どこで止めるかが判断の分かれ目**です。0.97117 という数字が「本物か過剰適合か」の議論はここに起因します。

**初心者向け補足**: **多重共線性**とは、説明変数どうしが強く相関していて、回帰係数が一意に定まりにくく（＝わずかなデータの変化で大きく振れる）なる状態のこと。

In [ ]:
WEIGHTS = {
    # Szymon Kłapiński — 52 arrays
    "szymon__altview": -0.03324829125939144,
    "szymon__digit_cat": 0.3220667472020567,
    "szymon__hgb": -0.2335617205707433,
    "szymon__imp_lgbm": 0.43541729289259135,
    "szymon__imp_lgbm_tuned": 0.6163303408425871,
    "szymon__imp_xgb": 0.13542168369070753,
    "szymon__imp_xgb_tuned": -0.10480878275278983,
    "szymon__lat_cat": -0.7438088044624854,
    "szymon__lat_lgbm": -1.0144303025834351,
    "szymon__lat_xgb": -0.6575515901966305,
    "szymon__latmax_lgbm": 0.04397209261460211,
    "szymon__latr1_lgbm": -0.11176812555263667,
    "szymon__latr1_xgb": 1.3031762062560392,
    "szymon__lattri_xgb": 0.4310110574194461,
    "szymon__latwide_cat": -0.006709823940269038,
    "szymon__latwide_lgbm": 0.35981725681450993,
    "szymon__latwide_xgb": -0.2381933803173623,
    "szymon__lgbm": -0.31107230683949616,
    "szymon__lgbm_tuned": -0.26456896662170765,
    "szymon__lookup": 0.816629346125691,
    "szymon__mlp": 0.13369621589618222,
    "szymon__naji01": -0.3355711870347737,
    "szymon__naji02": 0.5153340102089142,
    "szymon__naji04": -0.1418701798243691,
    "szymon__pub_cat": -0.688673603108497,
    "szymon__pub_donlgbm": 0.5435136049722966,
    "szymon__pub_evg": 0.22753306076239677,
    "szymon__pub_resnet": -0.5508594763080862,
    "szymon__pub_rmlp": 1.0134890389440583,
    "szymon__pub_tabm": -2.3802569521757713,
    "szymon__pub_tabnet": 0.8060802164978454,
    "szymon__pubfe_cat": -0.20772241766247027,
    "szymon__pubfe_lgb": 0.14879940319856277,
    "szymon__pubfe_xgb": 0.2848098869129542,
    "szymon__pubmk_cat": 0.5272388598283316,
    "szymon__pubmk_nn": 0.5467652036977798,
    "szymon__realmlp": -0.24956410390907388,
    "szymon__tabm_bounds": -0.004333750549997984,
    "szymon__tabm_deep": -0.4060712908581089,
    "szymon__tabm_deeper": 1.1520907329010912,
    "szymon__tabm_div": -0.4740264595186325,
    "szymon__tabm_imp": 0.4742838975553171,
    "szymon__tabm_wide": 0.4458716300984079,
    "szymon__tabm_x12": 0.013855621130342416,
    "szymon__view_bounds_lgbm": 0.512072385081101,
    "szymon__view_nolattice_lgbm": -0.232937116163811,
    "szymon__view_rank_lgbm": -0.46194889479778134,
    "szymon__view_resid_cat": -0.3306506920878424,
    "szymon__view_resid_lgbm": 0.11660774347749192,
    "szymon__view_resid_xgb": -0.3253830892632056,
    "szymon__xgb": 0.3014955472928735,
    "szymon__xgb_tuned": -0.3899215679030236,
    # AdarshAleti — 20 arrays
    "adarsh__catnative": 2.34638209510739,
    "adarsh__gcatd8": -0.49612067009569916,
    "adarsh__gcatlr02": -0.26501838357660923,
    "adarsh__gcatnote": 0.023286492731686197,
    "adarsh__gcatseed7": -0.544260038881982,
    "adarsh__glgb127": 0.20715424836635687,
    "adarsh__glgbcs3": -0.23882909915674763,
    "adarsh__glgbd4": 0.30159425187605426,
    "adarsh__glgbnote2": 0.5007486391101597,
    "adarsh__gnn_note": -0.09341649555188306,
    "adarsh__gnn_te": 0.12163994383067986,
    "adarsh__gnn_wide": -0.15401586710409004,
    "adarsh__gxgbcs4": 0.5936862727973605,
    "adarsh__gxgbd4": 0.2677628558327892,
    "adarsh__hgbte": -0.16123763247569164,
    "adarsh__lgbnote": -0.1110303983926198,
    "adarsh__lgbs7": -0.09593181613260451,
    "adarsh__lgbte": 0.3224456687092184,
    "adarsh__logregte": 0.040909486286174725,
    "adarsh__xgbte": 0.6461148359920936,
    # Darius Hafshar — 6 arrays
    "golem__b": -0.3635033816121735,
    "golem__c": -0.13720906599072882,
    "golem__d": -0.6079986963345088,
    "golem__e": -0.24238729945527163,
    "golem__f": -0.14731005783715106,
    "golem__g": 0.009447464875351658,
    # Rayk Kretzschmar — 5 arrays
    "rayk__fmdeep": 0.29790407454733586,
    "rayk__fmnum": 1.2700260823660297,
    "rayk__fmplr": 0.1831743485627224,
    "rayk__fmpure": -0.05781492440915468,
    "rayk__fmwide": -0.609727023720551,
    # boltuzamaki — 1 arrays
    "bolt__lookup_v2_s42": 0.6402077846362803,
    # Naji — 1 arrays
    "najiama__blend19": 7.98767600786824,
}


## 【解説】配列のロードと順位パーセンタイル変換

**What**: 2つの関数を定義します。

- `load_test(name)`: `"szymon__lat_lgbm"` のような名前を `作者__配列名` に分解し、作者ごとに違う保存形式（`.npy`, `.parquet`, `.csv`）から該当する予測を読み、**`sub_ids` の順に並べ替えて**返す。
- `rank_pct(v)`: 値を**順位パーセンタイル**（0〜1）に変換する。

**Why**:

- **`load_test` の設計**: 名前を `src__stem` に分けるだけで読み込み方が決まる、という規約にしておくと、`WEIGHTS` に新しいエントリを1行足すだけで新しいモデルを追加できます。**設定（重み）とロジック（読み方）を分離**する良い設計です。`_bolt` をグローバルにキャッシュしているのは、parquet の読み込みが重いので1回だけにするため。
- **`.loc[sub_ids]` の意味**: `set_index("id").loc[sub_ids]` で、**IDをキーに提出順へ並べ替え**ています。「たまたま同じ順番だろう」と仮定せず明示的に揃えるのが重要で、ここを怠ると気付かないまま完全に無意味な提出ができます（形式は正しく、エラーも出ない — 最も見つけにくいバグの типа）。
- **`rank_pct` の意味**: `rank(method="average", pct=True)` は「同値には平均順位を与え、結果を 0〜1 に正規化」します。これを通すと、どんな分布の予測値も**一様分布**になります。上の「評価指標」で述べた通り、AUCは順位しか見ないので、これは情報を捨てずにスケールだけ揃える操作です。

**初心者向け補足**: `method="average"` は同点処理の方式。たとえば3位タイが2つあれば両方に 3.5 を与えます。`"min"` や `"first"` にすると同点の扱いが変わり、AUCが微妙に変動することがあります。

In [ ]:
NPY = {"szymon": SZ, "adarsh": AD, "golem": GO, "rayk": RK}
_bolt = None


def load_test(name):
    """Return this array's test-side prediction, aligned to sample_submission order."""
    global _bolt
    src, stem = name.split("__", 1)
    if src in NPY:
        return np.load(f"{NPY[src]}/test_{stem}.npy").astype(np.float64)
    if src == "bolt":
        if _bolt is None:
            _bolt = pd.read_parquet(f"{BO}/test_predictions.parquet").set_index("id").loc[sub_ids]
        return _bolt[stem].to_numpy(dtype=np.float64)
    if src == "najiama":
        naji = pd.read_csv(f"{NA}/19_blend_submission.csv.csv").set_index("id").loc[sub_ids]
        return naji["addicted_label"].to_numpy(dtype=np.float64)
    raise KeyError(name)


def rank_pct(v):
    return pd.Series(v).rank(method="average", pct=True).to_numpy(dtype=np.float64)


## 【解説】ブレンドの実行

**What**: 自作の1列（すでに重み込みで計算済み）を初期値とし、`WEIGHTS` の各エントリについて `重み × rank_pct(その配列)` を足し込んでいきます。

**Why**:

- **順位に変換してから重みを掛ける順序が重要**です。`w * rank_pct(v)` であって `rank_pct(w * v)` ではありません。後者では重みが順位を変えないので（正の重みなら単調変換）、ブレンドとして機能しません。
- **`float64` で計算している理由**: 85項の加算で、しかも正負が打ち消し合う項が多い。`float32` だと**桁落ち**（大きな数どうしの引き算で有効数字が失われる）が起きうるので、倍精度で持つのが安全です。
- **なぜ自作の列だけ先に足しているか**: 原著者の自前モデルは非公開で、すでに重みを掛けた状態の1列としてデータセット化されています。戦略を隠しつつ結果は共有する、というPlaygroundでよくある折衷です。

**初心者向け補足**: 累積加算を `score += ...` で書いているので、`score` は最初から NumPy 配列である必要があります。`.to_numpy(dtype=np.float64)` で明示的に変換しているのはそのためです。

In [ ]:
# my own contribution, precomputed and already weighted
score = (pd.read_csv(f"{OURS}/s6e8_our_component.csv")
           .set_index("id").loc[sub_ids, "our_component"].to_numpy(dtype=np.float64))

for name, w in WEIGHTS.items():
    score += w * rank_pct(load_test(name))

print(f"blended {len(WEIGHTS)} public arrays + 1 own column")


## 【解説】提出ファイルの書き出し

**What**: 最終スコアをもう一度 `rank_pct` に通してから `submission.csv` に書き出します。

**Why**: 最後にもう一度順位化しているのは主に**見た目と安全のため**です。AUCは単調変換に不変なので、この操作でスコアは1ミリも変わりません。ただし
- 出力が必ず 0〜1 に収まるので、確率を期待する検証ロジックに弾かれない、
- 負の重みの影響で生の `score` が負値や大きな値になっていても問題にならない、
という実務上の利点があります。

**「なぜこの1行が入っているのか」を考えることが大事**です。AUCコンペで「最後に順位化しておけば損はしない」というのは覚えておいて良い定石ですが、**RMSEやLogLossのコンペで同じことをすると壊滅的に悪化します**（そちらは値そのものが評価されるため）。指標を見てから後処理を決める、という順序を常に守ってください。

In [ ]:
submission = pd.DataFrame({"id": sub_ids, "addicted_label": rank_pct(score)})
submission.to_csv("submission.csv", index=False)
print(submission.shape)
submission.head()
